# <font color=#c51b8a>VPOD 'Mine-n-Match':</font>
## <font color=#c994c7>Part 1 </font> - Use Species Names from Merged Accessory DBs to Query NCBI for All Related Opsin Sequences  

In [1]:
import os
import json
import pandas as pd
email = 'seth.frazer@embl.de'

In [2]:
from mnm_scripts.mine_n_match_functions import  merge_accessory_dbs
from mnm_scripts.ncbi_functions import ncbi_fetch_opsins
from mnm_scripts.utils import fasta_to_dataframe

## <font color=#c51b8a>Load data-tables with all of the species and Lambda Max data from accessory lmax databases</font> 

### <font color=#c994c7>VPOD Single Cell Microspectrophotmetry (SCP) Datatable </font>
### In this case our dataframe does not have full species name in one column so we must create a list by directly combining the genus and species names. Then filter to create a list of all unique species names 


In [3]:
report_dir = './circularity_test'
df_list = []

In [4]:
scp_file = f'{report_dir}/sampled_exclusive_wt_20260311_130205.tsv'
scp_df = pd.read_csv(scp_file, index_col=0, sep='\t')
scp_df['Full_Species'] = scp_df['Full_Species'].str.replace('_',' ')
df_list.append(scp_df)
scp_df.head()

,LambdaMax,Accession,Opsin_Family,Full_Species,Genus,Species,Phylum,Class,Protein,RefId
Seq_Id,,,,,,,,,,
S234,358.0,U63972.1,SWS1,Rattus norvegicus,Rattus,norvegicus,Chordata,Mammalia,MSGEXEFYLFQNISSVGPWDGPQYHIAPVWAFHLQAAFMGFVFFAG...,132.0
S684,507.0,OP722950.1,IV-LWS,Chrysochroa mniszechii,Chrysochroa,mniszechii,Arthropoda,Insecta,MSALGEPNFAAWSAQRVMSGAFGGNYTVVDKVPPEMLYLVDHHWYQ...,418.0
S310,516.0,AF132042.1,MWS,Cavia porcellus,Cavia,porcellus,Chordata,Mammalia,MAQRWGPHALSGVQAQDAYEDSTQASLFTYTNSNNTRGPFEGPNYH...,155.0
S729,532.0,OK930069.1,IV-LWS,Automeris io,Automeris,io,Arthropoda,Insecta,MTISLDPGPGLAALQAWGGQVAAYGAANQTVVDKVPPDMLHMVDAH...,416.0
S436,486.0,LC260050,Rh2,Oryzias luzonensis,Oryzias,luzonensis,Chordata,Actinopteri,MGWDGGEQNGTEGKNFYIPMSNRTGVVRSPYEYPQYYMVDPIMFKI...,367.0


In [5]:
scp_df.drop(columns='Accession', inplace=True)
scp_df.head()

,LambdaMax,Opsin_Family,Full_Species,Genus,Species,Phylum,Class,Protein,RefId
Seq_Id,,,,,,,,,
S234,358.0,SWS1,Rattus norvegicus,Rattus,norvegicus,Chordata,Mammalia,MSGEXEFYLFQNISSVGPWDGPQYHIAPVWAFHLQAAFMGFVFFAG...,132.0
S684,507.0,IV-LWS,Chrysochroa mniszechii,Chrysochroa,mniszechii,Arthropoda,Insecta,MSALGEPNFAAWSAQRVMSGAFGGNYTVVDKVPPEMLYLVDHHWYQ...,418.0
S310,516.0,MWS,Cavia porcellus,Cavia,porcellus,Chordata,Mammalia,MAQRWGPHALSGVQAQDAYEDSTQASLFTYTNSNNTRGPFEGPNYH...,155.0
S729,532.0,IV-LWS,Automeris io,Automeris,io,Arthropoda,Insecta,MTISLDPGPGLAALQAWGGQVAAYGAANQTVVDKVPPDMLHMVDAH...,416.0
S436,486.0,Rh2,Oryzias luzonensis,Oryzias,luzonensis,Chordata,Actinopteri,MGWDGGEQNGTEGKNFYIPMSNRTGVVRSPYEYPQYYMVDPIMFKI...,367.0


### <font color=#c994c7>Merge Accessory Lambda Max Databases</font>

In [6]:
# Call the function to merge all the species, lambdamax, and potential accession information into one dataframe
merged_df, merged_df_file = merge_accessory_dbs(df_list, report_dir)
merged_df.head()

,Full_Species,Accession,Seq_Id,LambdaMax
comp_db_id,,,,
0,Rattus norvegicus,None,S234,358.0
1,Chrysochroa mniszechii,None,S684,507.0
2,Cavia porcellus,None,S310,516.0
3,Automeris io,None,S729,532.0
4,Oryzias luzonensis,None,S436,486.0


In [7]:
merged_df_file

'./circularity_test/VPOD_in_vivo_1.0_2026-03-13_13-13-51.csv'

In [8]:
merged_df.shape

(50, 4)

### <font color=#c994c7>All unique species names have been extracted from accessory databases. Now we iteratively query NCBI for opsins from each species.</font>

In [9]:
# Option to just load an existing merged database and by-pass replication
#report_dir = './data_sources/lmax/'
#merged_df = pd.read_csv(f'{report_dir}/VPOD_in_vivo_1.0_2025-09-22_15-52-28.csv')

In [10]:
species_list = merged_df["Full_Species"].to_list()
# This is the length of the species list before filtering for only unique species names
len(species_list)

50

In [11]:
# This is the length of the species list which only includes unique species names
species_list = list(set(species_list))
len(species_list)

38

In [12]:
species_list

['Cyprichromis pavo',
 'Gallus gallus',
 'Taeniopygia guttata',
 'Sargocentron diadema',
 'Carassius auratus',
 'Cyprichromis leptosoma',
 'latipes latipes',
 'Trematocara unimaculatum',
 'Eumaeus atala',
 'Poecilia reticulata',
 'Metriaclima zebra',
 'Callophrys sheridanii',
 'Xenopeltis unicolor',
 'Odocoileus virginianus',
 'Dermochelys coriacea',
 'Dimidiochromis compressiceps',
 'Danaus plexippus',
 'Aotus azaraiboliviensis',
 'Greenwoodochromis bellcrossi',
 'Cavia porcellus',
 'Danio rerio',
 'Haplotaxodon microlepis',
 'Grus americana',
 'Oryzias luzonensis',
 'Rattus norvegicus',
 'Columba livia',
 'Lithobates pipiens',
 'Paracyprichromis brieni',
 'Apis cerana',
 'Anolis carolinensis',
 'Epargyreus clarus',
 'Myripristis violacea',
 'Chrysochroa mniszechii',
 'Macroglossum stellatarum',
 'Verasper variegatus',
 'Oryzias minutillus',
 'Oreochromis niloticus',
 'Automeris io']

## <font color=#c51b8a>Run NCBI Query Mining Process</font>

In [13]:
ncbi_query_df, query_report_dir = ncbi_fetch_opsins(email=email, job_label='all_dbs', out='all_dbs', species_list=species_list)

Creating Job Directory

Saving Species Query List to Text

Constructing Taxon Dictionary, Including Species Synonyms

Existing Taxon Dictionary Found! Checking if we need to update it...

No need to update! Carry on :) 

Starting Queries to NCBI for Opsin Sequences



Processing species queries: 100%|█████████████████████████| 38/38 [00:29<00:00]


NCBI Queries Complete!
Now Extracting and Formatting Results For DataFrame...



Formatting species queries: 100%|█████████████████████████| 38/38 [00:00<00:00]

DataFrame Formatted and Saved to ./mnm_data/mnm_on_all_dbs_2026-03-13_13-13-52/all_dbs_ncbi_q_data.csv

FASTA File Saved...

Saving txt file with names of species that retrieved no results for opsins...

Saving txt file with names of species that retrieved results for opsins but are NOT in submitted species list...

Saving and returning cleaned dataframe with only species entries from species list...

Saving another dataframe with species that retrieved results for opsins but are NOT in submitted species list for further examination...

Clean FASTA File Saved...



In [14]:
ncbi_query_df.shape

(290, 11)

In [15]:
len(ncbi_query_df['Full_Species'].unique())

32

## <font color=#c51b8a>Load Accessory Opsin Sequence Databases</font> 

### <font color=#c994c7>Load Previous MnM Data</font>

In [16]:
# Ignore this box, it's here for cases where you want to load existing query df
#query_report_dir = "mnm_data/mnm_on_all_dbs_2025-10-03_17-37-05"
#ncbi_query_file = f'{query_report_dir}/mnm_on_all_dbs_ncbi_q_data_cleaned.csv'
#ncbi_query_df = pd.read_csv(ncbi_query_file)

In [17]:
import json
taxon_file = './data_sources/taxonomy/ncbi_taxon_dict.json'
with open(taxon_file, 'r') as f:
    existing_taxon_dict = json.load(f)

In [18]:
ncbi_sp_list = ncbi_query_df['Full_Species'].to_list()
ncbi_prot_list = ncbi_query_df['Protein'].to_list()
ncbi_query_df.head()

,Accession,Phylum,Subphylum,Class,Genus,Species,Full_Species,Protein,Gene_Description,Species_Synonym_Used,Prot_Len
0,BAJ60904.1,Chordata,Craniata,Actinopteri,Cyprichromis,pavo,Cyprichromis pavo,MNGTEGPFFYVPMANTTGIVRSPYEYPQHYLVNPAAYAALGAYMFF...,rod opsin [Cyprichromis pavo],NA,354
1,P51475.1,Chordata,Craniata,Aves,Gallus,gallus,Gallus gallus,MSSNSSQAPPNGTPGPFDGPQWPYQAPQSTYVGVAVLMGTVVACAS...,RecName: Full=Pinopsin; AltName: Full=Pineal g...,NA,351
2,P22329.1,Chordata,Craniata,Aves,Gallus,gallus,Gallus gallus,MAAWEAAFAARRRHEEEDTTRDSVFTYTNSNNTRGPFEGPNYHIAP...,RecName: Full=Red-sensitive opsin; AltName: Fu...,NA,362
3,P28683.1,Chordata,Craniata,Aves,Gallus,gallus,Gallus gallus,MNGTEGINFYVPMSNKTGVVRSPFEYPQYYLAEPWKYRLVCCYIFF...,RecName: Full=Green-sensitive opsin; AltName: ...,NA,355
4,NP_990769.1,Chordata,Craniata,Aves,Gallus,gallus,Gallus gallus,MSSDDDFYLFTNGSVPGPWDGPQYHIAPPWAFYLQTAFMGIVFAVG...,violet-sensitive opsin [Gallus gallus],NA,347


In [19]:
report_dir2 = report_dir
sequence_list = []
acc_list = []
db_sp_list = []
prot_descriptions = []
source_list = []

### <font color=#c994c7>Extracting Sequence Meta-Data</font>

In [20]:
scp_df.head()

,LambdaMax,Opsin_Family,Full_Species,Genus,Species,Phylum,Class,Protein,RefId,Accession
Seq_Id,,,,,,,,,,
S234,358.0,SWS1,Rattus norvegicus,Rattus,norvegicus,Chordata,Mammalia,MSGEXEFYLFQNISSVGPWDGPQYHIAPVWAFHLQAAFMGFVFFAG...,132.0,<NA>
S684,507.0,IV-LWS,Chrysochroa mniszechii,Chrysochroa,mniszechii,Arthropoda,Insecta,MSALGEPNFAAWSAQRVMSGAFGGNYTVVDKVPPEMLYLVDHHWYQ...,418.0,<NA>
S310,516.0,MWS,Cavia porcellus,Cavia,porcellus,Chordata,Mammalia,MAQRWGPHALSGVQAQDAYEDSTQASLFTYTNSNNTRGPFEGPNYH...,155.0,<NA>
S729,532.0,IV-LWS,Automeris io,Automeris,io,Arthropoda,Insecta,MTISLDPGPGLAALQAWGGQVAAYGAANQTVVDKVPPDMLHMVDAH...,416.0,<NA>
S436,486.0,Rh2,Oryzias luzonensis,Oryzias,luzonensis,Chordata,Actinopteri,MGWDGGEQNGTEGKNFYIPMSNRTGVVRSPYEYPQYYMVDPIMFKI...,367.0,<NA>


In [21]:
scp_df_filterd = scp_df[~scp_df['Protein'].isin(ncbi_prot_list)]
scp_df_filterd = scp_df_filterd.reset_index(drop=True)
scp_df_filterd['source'] = 'test_data'
scp_df_filterd.shape

(17, 11)

In [22]:
scp_df_filterd.to_csv(f'{report_dir2}/test_data_filtered.csv')
scp_df_filterd.head()

,LambdaMax,Opsin_Family,Full_Species,Genus,Species,Phylum,Class,Protein,RefId,Accession,source
0,358.0,SWS1,Rattus norvegicus,Rattus,norvegicus,Chordata,Mammalia,MSGEXEFYLFQNISSVGPWDGPQYHIAPVWAFHLQAAFMGFVFFAG...,132.0,<NA>,test_data
1,539.0,LWS,Aotus azaraiboliviensis,Aotus,azaraiboliviensis,Chordata,Lepidosauria,MAQQWSLQRLAGRHPQDNHEDSTQSSIFTYTNSNSTRGPFEGPNYH...,384.0,<NA>,test_data
2,498.0,Rh1,Cyprichromis leptosoma,Cyprichromis,leptosoma,Chordata,Actinopteri,MANTTGIVRSPYEYPQHYLVNPAAYAALGAYMFFLMLVGFPINFLT...,163.0,<NA>,test_data
3,488.0,Rh1,Greenwoodochromis bellcrossi,Greenwoodochromis,bellcrossi,Chordata,Actinopteri,MTNTTGVIRSPYEYPQHYLVSPAAYAALGAYMFFLIIVGFPINFLT...,163.0,<NA>,test_data
4,502.0,Rh1,Dimidiochromis compressiceps,Dimidiochromis,compressiceps,Chordata,Actinopteri,MVNTTGIVRSPYEYPQHYLVSPAAYAALGAYMFFLILVGFPINFLT...,163.0,<NA>,test_data


In [23]:
sequence_list += scp_df_filterd['Protein'].to_list()
acc_list += scp_df_filterd['Accession'].to_list()
db_sp_list += scp_df_filterd['Full_Species'].to_list()
prot_descriptions += scp_df_filterd['Opsin_Family'].to_list()
source_list += scp_df_filterd['source'].to_list()

### <font color=#c994c7>Extract all taxon info for the species these collection of sequences belong to:</font>


In [24]:
gn_list = []
sp_list = []
for sp in db_sp_list:
#    print(sp)
    gn_list.append(sp.split(' ', 1)[0])
    sp_list.append(sp.split(' ', 1)[1])

In [25]:
# Load our existing taxonomy dictionary and pull relevant taxon info
taxon_file = './data_sources/taxonomy/ncbi_taxon_dict.json'
if os.path.isfile(taxon_file):
    with open(taxon_file, 'r') as f:
        species_taxon_dict = json.load(f)
        
phylum_list = []
subphylum_list = []
class_list = []      
for sp in db_sp_list:
    phylum_list.append(species_taxon_dict[sp]["Phylum"])
    subphylum_list.append(species_taxon_dict[sp]["Subphylum"])
    class_list.append(species_taxon_dict[sp]["Class"])

### <font color=#c994c7>Create Merged Dataframe From All Accessory Opsin Sequence DBs</font>

In [26]:
#make a merged df of all the accessory seq dbs, filter out reedundant datapoints, then append to the end of the NCBI query sheet?
data = {'Accession': acc_list, 'Phylum': phylum_list, 'Subphylum': subphylum_list, 'Class': class_list,'Genus': gn_list, 'Species': sp_list, 'Full_Species': db_sp_list, 'Gene_Description': prot_descriptions, 'Protein' : sequence_list, 'source' : source_list} 
#data = {'Accession': acc_list, 'Genus': gn_list, 'Species': sp_list, 'Full_Species': db_sp_list, 'Gene_Description': prot_descriptions, 'Protein' : sequence_list, 'source' : source_list} 

acc_seq_db_df = pd.DataFrame(data)
acc_seq_db_df.head()

,Accession,Phylum,Subphylum,Class,Genus,Species,Full_Species,Gene_Description,Protein,source
0,<NA>,Chordata,Craniata,Mammalia,Rattus,norvegicus,Rattus norvegicus,SWS1,MSGEXEFYLFQNISSVGPWDGPQYHIAPVWAFHLQAAFMGFVFFAG...,test_data
1,<NA>,Unknown,Unknown,Unknown,Aotus,azaraiboliviensis,Aotus azaraiboliviensis,LWS,MAQQWSLQRLAGRHPQDNHEDSTQSSIFTYTNSNSTRGPFEGPNYH...,test_data
2,<NA>,Chordata,Craniata,Actinopteri,Cyprichromis,leptosoma,Cyprichromis leptosoma,Rh1,MANTTGIVRSPYEYPQHYLVNPAAYAALGAYMFFLMLVGFPINFLT...,test_data
3,<NA>,Chordata,Craniata,Actinopteri,Greenwoodochromis,bellcrossi,Greenwoodochromis bellcrossi,Rh1,MTNTTGVIRSPYEYPQHYLVSPAAYAALGAYMFFLIIVGFPINFLT...,test_data
4,<NA>,Chordata,Craniata,Actinopteri,Dimidiochromis,compressiceps,Dimidiochromis compressiceps,Rh1,MVNTTGIVRSPYEYPQHYLVSPAAYAALGAYMFFLILVGFPINFLT...,test_data


In [27]:
acc_seq_db_df.shape

(17, 10)

In [28]:
acc_seq_db_df_filtered = acc_seq_db_df.copy()
acc_seq_db_df_filtered.drop_duplicates(subset=['Full_Species', 'Protein'],  keep='first', inplace=True)
acc_seq_db_df_filtered=acc_seq_db_df_filtered.reset_index(drop=True)
acc_seq_db_df_filtered.shape

(17, 10)

In [29]:
len(set(acc_seq_db_df_filtered['Full_Species'].to_list()))

14

In [30]:
acc_seq_db_df_filtered.to_csv(f'{report_dir2}/vpod_comp_accessory_seq_dbs.csv')

In [31]:
fasta_file = f'./{report_dir2}/acc_db_seqs.fasta'
with open(fasta_file, 'w') as f:
    for id, seq in zip(acc_seq_db_df_filtered['Accession'], acc_seq_db_df_filtered['Protein']):
        f.write(f'>{id}\n{seq}\n')

### <font color=#c994c7>Merge the Formated Accessory Sequence DBs w/the Mined NCBI Data</font>

In [32]:
ncbi_query_df.head()

,Accession,Phylum,Subphylum,Class,Genus,Species,Full_Species,Protein,Gene_Description,Species_Synonym_Used,Prot_Len
0,BAJ60904.1,Chordata,Craniata,Actinopteri,Cyprichromis,pavo,Cyprichromis pavo,MNGTEGPFFYVPMANTTGIVRSPYEYPQHYLVNPAAYAALGAYMFF...,rod opsin [Cyprichromis pavo],NA,354
1,P51475.1,Chordata,Craniata,Aves,Gallus,gallus,Gallus gallus,MSSNSSQAPPNGTPGPFDGPQWPYQAPQSTYVGVAVLMGTVVACAS...,RecName: Full=Pinopsin; AltName: Full=Pineal g...,NA,351
2,P22329.1,Chordata,Craniata,Aves,Gallus,gallus,Gallus gallus,MAAWEAAFAARRRHEEEDTTRDSVFTYTNSNNTRGPFEGPNYHIAP...,RecName: Full=Red-sensitive opsin; AltName: Fu...,NA,362
3,P28683.1,Chordata,Craniata,Aves,Gallus,gallus,Gallus gallus,MNGTEGINFYVPMSNKTGVVRSPFEYPQYYLAEPWKYRLVCCYIFF...,RecName: Full=Green-sensitive opsin; AltName: ...,NA,355
4,NP_990769.1,Chordata,Craniata,Aves,Gallus,gallus,Gallus gallus,MSSDDDFYLFTNGSVPGPWDGPQYHIAPPWAFYLQTAFMGIVFAVG...,violet-sensitive opsin [Gallus gallus],NA,347


In [33]:
acc_seq_db_df_filtered.drop(columns='source', inplace =True)
acc_seq_db_df_filtered.head()

,Accession,Phylum,Subphylum,Class,Genus,Species,Full_Species,Gene_Description,Protein
0,<NA>,Chordata,Craniata,Mammalia,Rattus,norvegicus,Rattus norvegicus,SWS1,MSGEXEFYLFQNISSVGPWDGPQYHIAPVWAFHLQAAFMGFVFFAG...
1,<NA>,Unknown,Unknown,Unknown,Aotus,azaraiboliviensis,Aotus azaraiboliviensis,LWS,MAQQWSLQRLAGRHPQDNHEDSTQSSIFTYTNSNSTRGPFEGPNYH...
2,<NA>,Chordata,Craniata,Actinopteri,Cyprichromis,leptosoma,Cyprichromis leptosoma,Rh1,MANTTGIVRSPYEYPQHYLVNPAAYAALGAYMFFLMLVGFPINFLT...
3,<NA>,Chordata,Craniata,Actinopteri,Greenwoodochromis,bellcrossi,Greenwoodochromis bellcrossi,Rh1,MTNTTGVIRSPYEYPQHYLVSPAAYAALGAYMFFLIIVGFPINFLT...
4,<NA>,Chordata,Craniata,Actinopteri,Dimidiochromis,compressiceps,Dimidiochromis compressiceps,Rh1,MVNTTGIVRSPYEYPQHYLVSPAAYAALGAYMFFLILVGFPINFLT...


In [34]:
final_query_df = pd.concat([ncbi_query_df, acc_seq_db_df_filtered]).reset_index(drop = True)
final_query_df['Prot_Len'] = final_query_df['Protein'].str.len()
final_query_df.to_csv(f'{query_report_dir}/ncbi_q_merged_w_acc_seq_db.csv', index=False)
final_query_df.head()

,Accession,Phylum,Subphylum,Class,Genus,Species,Full_Species,Protein,Gene_Description,Species_Synonym_Used,Prot_Len
0,BAJ60904.1,Chordata,Craniata,Actinopteri,Cyprichromis,pavo,Cyprichromis pavo,MNGTEGPFFYVPMANTTGIVRSPYEYPQHYLVNPAAYAALGAYMFF...,rod opsin [Cyprichromis pavo],NA,354
1,P51475.1,Chordata,Craniata,Aves,Gallus,gallus,Gallus gallus,MSSNSSQAPPNGTPGPFDGPQWPYQAPQSTYVGVAVLMGTVVACAS...,RecName: Full=Pinopsin; AltName: Full=Pineal g...,NA,351
2,P22329.1,Chordata,Craniata,Aves,Gallus,gallus,Gallus gallus,MAAWEAAFAARRRHEEEDTTRDSVFTYTNSNNTRGPFEGPNYHIAP...,RecName: Full=Red-sensitive opsin; AltName: Fu...,NA,362
3,P28683.1,Chordata,Craniata,Aves,Gallus,gallus,Gallus gallus,MNGTEGINFYVPMSNKTGVVRSPFEYPQYYLAEPWKYRLVCCYIFF...,RecName: Full=Green-sensitive opsin; AltName: ...,NA,355
4,NP_990769.1,Chordata,Craniata,Aves,Gallus,gallus,Gallus gallus,MSSDDDFYLFTNGSVPGPWDGPQYHIAPPWAFYLQTAFMGIVFAVG...,violet-sensitive opsin [Gallus gallus],NA,347


In [35]:
final_query_df.shape

(307, 11)

In [36]:
fasta_file = f'./{query_report_dir}/mined_and_acc_seqs.fasta'
with open(fasta_file, 'w') as f:
    for id, seq in zip(final_query_df['Accession'], final_query_df['Protein']):
        f.write(f'>{id}\n{seq}\n')

## <font color=#c994c7>Part 2: Blast Filtering</font> - Filter non-visual opsins by using BLAST against our sequence database of visual and non-visual opsins.

- We  filter based on if the closest match is tagged as a non-visual or visual opsin.

In [37]:
from mnm_scripts.blastp import run_blastp_analysis
import pathlib
script_path = pathlib.Path().resolve()  # Get absolute path
wrk_dir = str(script_path).replace('\\', '/')

In [38]:
#query_report_dir = 'mnm_data/mnm_on_all_dbs_2025-10-03_17-37-05'
#fasta_file = f'./{query_report_dir}/mined_and_acc_seqs.fasta'

blast_db_path = './data_sources/blastdbs/mnm_opsin_ref_db'
blast_output = f'./{query_report_dir}/blast_reference_report'

In [39]:
blast_analysis_df = run_blastp_analysis('blastp', fasta_file, blast_db_path, blast_output, wrk_dir=wrk_dir)

INFO: Total records in input: 307. Unique uncached sequences to process: 1.
INFO: Running blastp for new sequences...

✅ Analysis complete. Filtered BLAST results saved to './mnm_data/mnm_on_all_dbs_2026-03-13_13-13-52/blast_reference_report_filtered.csv'


In [40]:
blast_analysis_df.head()

,qseqid,sseqid,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore
0,BAJ60904.1,visual_opsin_S47,100.00,354,0,0,1,354,1,354,0.0,728.0
1,P51475.1,NP_990740_Gallus_gallus_pinopsin,99.43,351,2,0,1,351,1,351,0.0,714.0
2,P22329.1,visual_opsin_S130,100.00,362,0,0,1,362,1,362,0.0,745.0
3,P28683.1,visual_opsin_S121,100.00,355,0,0,1,355,1,355,0.0,736.0
4,NP_990769.1,visual_opsin_S92,100.00,347,0,0,1,347,1,347,0.0,712.0


Note, the size of our dataframe will decrease here because any sequences which returned a 'blast unsuccessful' is dropped since they are likely not even a member of the larger opsin protein family

In [41]:
blast_analysis_df.shape

(306, 12)

Now let's clean the blast dataframe to keep only the accessions/sequences matched to visual opsins

In [42]:
clean_blast_analysis_df = blast_analysis_df.copy()
clean_blast_analysis_df = clean_blast_analysis_df[(clean_blast_analysis_df['sseqid'].str.contains('visual_opsin')) & (clean_blast_analysis_df['pident'] >= 20)]
clean_blast_analysis_df.shape

(244, 12)

Now we filter the main dataframe with the accessions matchd to visual opsins in the blast dataframe
- There will be more sequences in the main dataframe because there is a certain amount of permitted overlap

In [43]:
import pandas as pd
final_query_df = pd.read_csv(f'./{query_report_dir}/ncbi_q_merged_w_acc_seq_db.csv')
final_query_df.head()

,Accession,Phylum,Subphylum,Class,Genus,Species,Full_Species,Protein,Gene_Description,Species_Synonym_Used,Prot_Len
0,BAJ60904.1,Chordata,Craniata,Actinopteri,Cyprichromis,pavo,Cyprichromis pavo,MNGTEGPFFYVPMANTTGIVRSPYEYPQHYLVNPAAYAALGAYMFF...,rod opsin [Cyprichromis pavo],NaN,354
1,P51475.1,Chordata,Craniata,Aves,Gallus,gallus,Gallus gallus,MSSNSSQAPPNGTPGPFDGPQWPYQAPQSTYVGVAVLMGTVVACAS...,RecName: Full=Pinopsin; AltName: Full=Pineal g...,NaN,351
2,P22329.1,Chordata,Craniata,Aves,Gallus,gallus,Gallus gallus,MAAWEAAFAARRRHEEEDTTRDSVFTYTNSNNTRGPFEGPNYHIAP...,RecName: Full=Red-sensitive opsin; AltName: Fu...,NaN,362
3,P28683.1,Chordata,Craniata,Aves,Gallus,gallus,Gallus gallus,MNGTEGINFYVPMSNKTGVVRSPFEYPQYYLAEPWKYRLVCCYIFF...,RecName: Full=Green-sensitive opsin; AltName: ...,NaN,355
4,NP_990769.1,Chordata,Craniata,Aves,Gallus,gallus,Gallus gallus,MSSDDDFYLFTNGSVPGPWDGPQYHIAPPWAFYLQTAFMGIVFAVG...,violet-sensitive opsin [Gallus gallus],NaN,347


In [44]:
final_query_df.shape

(307, 11)

In [45]:
seqs_to_keep = clean_blast_analysis_df['qseqid'].to_list()
non_visual_query_df = final_query_df.copy()

In [46]:
exclusion_patterns = [
    r'\b(rh1_)?exorh',                  # Matches 'exorh', 'exorha', 'exorh_1'
    r'\btmt([1-9])?',           # Matches 'tmt', 'tmt1a', 'tmt2_b'
    r'\bmelanopsin',             # Matches 'melanopsin', 'melanopsin_a'
    r'\bop(?:n|sin)\s?[3-9]',    # Matches 'opn4m_1', 'opsin 5b', 'opn3' etc.
    r'\bneur',                   # Matches 'neur', 'neuropsin', 'neural'
    r'\bchain\b'
]

In [47]:
# Join the patterns into a single regex string
exclusion_regex = '|'.join(exclusion_patterns)
# Apply the streamlined filters
final_query_df = final_query_df[
    (final_query_df['Accession'].isin(seqs_to_keep)) &
    (~final_query_df['Gene_Description'].str.contains(exclusion_regex, case=False, regex=True)) &
    (final_query_df['Prot_Len'] <= 600) & (final_query_df['Prot_Len'] >= 300) &
    (~final_query_df['Full_Species'].str.contains(r'\bDaphnia\b', case=False, regex=True))
]
final_query_df.reset_index(inplace=True)

In [48]:
final_query_df.shape

(204, 12)

In [49]:
final_query_df.to_csv(f'{query_report_dir}/ncbi_q_merged_w_acc_seq_db_visual_only.csv')

fasta_file = f'./{query_report_dir}/mined_and_acc_seqs_visual_only.fasta'
with open(fasta_file, 'w') as f:
    for id, seq in zip(final_query_df['Accession'], final_query_df['Protein']):
        f.write(f'>{id}\n{seq}\n')

Let's also save all the non-visual opsin hits just for the sake of posterity

In [50]:
non_visual_blast_df = blast_analysis_df.copy()
non_visual_blast_df = non_visual_blast_df[~non_visual_blast_df['sseqid'].str.contains('visual_opsin')]
non_visual_blast_df.shape

(62, 12)

In [51]:
non_visual_blast_df

,qseqid,sseqid,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore
1,P51475.1,NP_990740_Gallus_gallus_pinopsin,99.430,351,2,0,1,351,1,351,0.000000e+00,714.0
7,NP_001296985.2,XP_023132487_Amphiprion_ocellaris_parapinopsin_a,33.333,291,180,6,19,301,20,304,2.410000e-46,160.0
8,NP_001156364.2,NP_001104634_opsin3_D_rerio,32.203,295,188,6,14,303,22,309,6.890000e-42,146.0
10,BBC27385.1,XP_044208099_Thunnus_albacares_parapinopsin_a,32.639,288,182,5,28,314,37,313,2.800000e-46,159.0
11,AGK25001.1,XP_040523482_opsin3_G_gallus,41.325,317,159,5,12,308,8,317,1.600000e-78,243.0
...,...,...,...,...,...,...,...,...,...,...,...,...
271,XP_005448551.1,XP_051958039_Xyrauchen_texanus_parapinopsin_b,31.802,283,183,4,21,302,28,301,2.290000e-42,148.0
272,XP_005448550.1,XP_051958039_Xyrauchen_texanus_parapinopsin_b,31.802,283,183,4,30,311,28,301,2.720000e-42,148.0
274,XP_005467499.1,XP_040523482_opsin3_G_gallus,41.196,301,160,3,45,331,34,331,1.880000e-76,238.0
275,XP_003453961.1,XP_028970850_Esox_lucius_parietopsin,25.000,220,157,5,7,223,40,254,3.510000e-08,50.8


In [52]:
seqs_to_keep = non_visual_blast_df['qseqid']

non_visual_query_df = non_visual_query_df[
    (non_visual_query_df['Accession'].str.contains('|'.join(seqs_to_keep))) &
    (non_visual_query_df['Gene_Description'].str.contains(exclusion_regex, case=False, regex=True))
]

In [53]:
#non_visual_query_df = non_visual_query_df[(non_visual_query_df['Accession'].str.contains('|'.join(seqs_to_keep))) & (non_visual_query_df['Gene_Description'].str.contains('exorh|exoRH|tmt1|tm2|tmt3|melanopsin|opn4|opn5|opn6|opn8|tmt|opn7|NEUR'))]

In [54]:
non_visual_query_df.to_csv(f'./{query_report_dir}/ncbi_q_merged_w_acc_seq_db_nonvisual_only.csv', index=False)

## <font color=#c994c7>Part 3: OPTICS Predictions</font> - Predict Lmax of all queried opsin sequences 

In [91]:
%reload_ext autoreload
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [56]:
import sys
# Get the absolute path to the 'optics' directory.  This is crucial for robustness.
optics_path = 'D:\\safra\\Github\\optics'
# Add the 'optics' directory to the Python path
if optics_path not in sys.path: # Avoid adding multiple times
    sys.path.append(optics_path)
# Now you can import modules from 'optics' as usual
from optics_predictions import run_optics_predictions

In [ ]:
#OPTICS Predictions
#query_report_dir = 'mnm_data/mnm_on_all_dbs_2025-10-01_14-51-49'
#fasta_file = f'./{query_report_dir}/mined_and_acc_seqs_visual_only.fasta'
optics_df, optics_pred_file = run_optics_predictions(input_sequence=fasta_file, pred_dir=f'./{query_report_dir}', output='circ_test',
                           model="whole-dataset", encoding_method='aa_prop', blastp=True,
                           iden_report='blastp_report', refseq='bovine',
                           bootstrap=False, bootstrap_num=100, visualize_bootstrap=False, n_jobs=10, tolerate_non_standard_aa=True)


Model Used:	whole-dataset
Encoding Method:	aa_prop
Bootstrap:	False


0 sequences were removed due to length constraints.
Found 204 valid sequences to process, corresponding to 204 unique sequences.

Prediction cache file successfully loaded.

204 unique sequences found in cache. Predicting 0 new unique sequences.

INFO: Found 142 sequences in BLASTp cache. Processing 62 new sequences.
INFO: Running BLASTp for all query sequences...
INFO: Starting MAFFT pairwise analysis to closest VPOD match for 62 sequences...


[Parallel(n_jobs=10)]: Using backend LokyBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done  56 out of  62 | elapsed:  1.4min remaining:    8.6s
[Parallel(n_jobs=10)]: Done  62 out of  62 | elapsed:  1.4min finished



✅ Analysis complete. BLASTp analysis results saved to './mnm_data/mnm_on_all_dbs_2026-03-13_13-13-52/optics_on_mnm_data_2026-03-13_13-14-35/blastp_report.csv'

Predictions Complete! Results are in: ./mnm_data/mnm_on_all_dbs_2026-03-13_13-13-52/optics_on_mnm_data_2026-03-13_13-14-35


In [58]:
optics_df.head()

,Names,Single_Prediction,%Identity_Nearest_VPOD_Sequence,Sequence_Length,Lmax_Hex_Color
0,BAJ60904.1,490.1,100.000,354,#00fffe
1,P22329.1,568.1,100.000,362,#dcff00
2,P28683.1,509.0,100.000,355,#00ff17
3,NP_990769.1,412.3,99.712,347,#6500bb
4,P28682.1,453.9,100.000,361,#005cff


In [59]:
optics_df.shape

(204, 5)

## <font color=#c994c7>Part 4: Matching Predictions to Physiology Data</font> - Match sequence to it's closest physiologically measured lmax value based on OPTICS predictions

In [92]:
import warnings
import pandas as pd
from deepBreaks.preprocessing import read_data
from mnm_scripts.mine_n_match_functions import mine_n_match
email = 'sethfrazer@ucsb.edu'
warnings.filterwarnings("ignore")
warnings.simplefilter("ignore")

#### <font color=#c994c7>Load NCBI Query Data</font>

In [115]:
query_report_dir = 'mnm_data/mnm_on_wds-50_circ_test_2026-03-11_13-39-21' #re-define the report directory if needed
ncbi_q_file = f'./{query_report_dir}/ncbi_q_merged_w_acc_seq_db_visual_only.csv'
ncbi = pd.read_csv(ncbi_q_file, index_col=0)
ncbi_filtered = ncbi[ncbi['Accession'].isin(optics_df['Names'])]
ncbi_filtered.reset_index(inplace=True)
ncbi_filtered.head()

,level_0,index,Accession,Phylum,Subphylum,Class,Genus,Species,Full_Species,Protein,Gene_Description,Species_Synonym_Used,Prot_Len
0,0,1,P22329.1,Chordata,Craniata,Aves,Gallus,gallus,Gallus gallus,MAAWEAAFAARRRHEEEDTTRDSVFTYTNSNNTRGPFEGPNYHIAP...,RecName: Full=Red-sensitive opsin; AltName: Fu...,NaN,362
1,1,2,P28683.1,Chordata,Craniata,Aves,Gallus,gallus,Gallus gallus,MNGTEGINFYVPMSNKTGVVRSPFEYPQYYLAEPWKYRLVCCYIFF...,RecName: Full=Green-sensitive opsin; AltName: ...,NaN,355
2,2,3,NP_990769.1,Chordata,Craniata,Aves,Gallus,gallus,Gallus gallus,MSSDDDFYLFTNGSVPGPWDGPQYHIAPPWAFYLQTAFMGIVFAVG...,violet-sensitive opsin [Gallus gallus],NaN,347
3,3,4,P28682.1,Chordata,Craniata,Aves,Gallus,gallus,Gallus gallus,MHPPRPTTDLPEDFYIPMALDAPNITALSPFLVPQTHLGSPGLFRA...,RecName: Full=Blue-sensitive opsin; AltName: F...,NaN,361
4,4,5,NP_001384426.1,Chordata,Craniata,Aves,Gallus,gallus,Gallus gallus,MNGTEGQDFYVPMSNKTGVVRSPFEYPQYYLAEPWKFSALAAYMFM...,rhodopsin [Gallus gallus],NaN,351


In [116]:
ncbi_filtered.shape

(221, 13)

#### <font color=#c994c7>Load OPTICS Predictions</font>

In [114]:
pred_dir = '.\mnm_data\mnm_on_wds-50_circ_test_2026-03-11_13-39-21\optics_on_mnm_data_2026-03-11_13-59-35'
optics_pred_file = f'{pred_dir}/mnm_data_predictions.tsv'
optics_df = pd.read_csv(optics_pred_file, sep='\t')
optics_df.head()

,Names,Single_Prediction,%Identity_Nearest_VPOD_Sequence,Sequence_Length,Lmax_Hex_Color
0,P22329.1,568.1,100.000,362,#dcff00
1,P28683.1,509.0,100.000,355,#00ff17
2,NP_990769.1,412.3,99.712,347,#6500bb
3,P28682.1,453.9,100.000,361,#005cff
4,NP_001384426.1,503.1,100.000,351,#00ff6d


In [107]:
optics.shape

(221, 5)

#### <font color=#c994c7>Load Lmax Compendium Data</font>

In [108]:
#source_file = './data_sources/lmax/VPOD_in_vivo_v1.0_2025-05-09_18-17-49.csv'
source_file = './circularity_test/VPOD_in_vivo_1.0_2026-03-11_14-17-25.csv'
comp_db = pd.read_csv(source_file,index_col=0)
comp_db.head()

,Full_Species,Accession,Seq_Id,LambdaMax
comp_db_id,,,,
0,Rattus norvegicus,NaN,S234,358.0
1,Chrysochroa mniszechii,NaN,S684,507.0
2,Cavia porcellus,NaN,S310,516.0
3,Automeris io,NaN,S729,532.0
4,Oryzias luzonensis,NaN,S436,486.0


#### <font color=#c994c7>Run Mine-n-Match Script!</font>

In [117]:
final_err_filtered_df = mine_n_match(email, query_report_dir, source_file, ncbi_filtered, optics_pred_file, out='vpod_in_vivo', err_filter = 10, per_identity_minimum=20, prediction_to_use='Single_Prediction', identity_filter=False)

There were 0 unmatched species


#### <font color=#c994c7>Check the final dataframe</font>

- Need to add a portion to the MNM backend which further filters for sequences matched to synonyms of the same species w/ same lmax. 

In [118]:
#final_mnm_file_name = './mnm_data/mnm_on_all_dbs_2025-02-24_16-29-54/mnm_on_vpod_in_vivo_results_fully_filtered.csv'
#final_err_filtered_df = pd.read_csv(final_mnm_file_name, index_col=0)
final_err_filtered_df.head()

,Accession,Phylum,Subphylum,Class,Genus,Species,Full_Species,%Identity_Nearest_VPOD_Sequence,prediction_value,LambdaMax,abs_diff,candidate_opsin_count,comp_db_id,Protein,Gene_Description
mnm_id,,,,,,,,,,,,,,,
0,P22329.1,Chordata,Craniata,Aves,Gallus,gallus,Gallus gallus,100.000,568.1,571.0,2.9,2,17,MAAWEAAFAARRRHEEEDTTRDSVFTYTNSNNTRGPFEGPNYHIAP...,RecName: Full=Red-sensitive opsin; AltName: Fu...
1,BAM74441.1,Chordata,Craniata,Actinopteri,Poecilia,reticulata,Poecilia reticulata,49.697,566.9,571.0,4.1,3,39,MAEEWGKQVFAARRHEDTTRGAAFTYTNSNHTKDPFEGPNYHIAPR...,long wave sensitive-1 opsin [Poecilia reticulata]
2,AB223051.1,Mollusca,Unknown,Gastropoda,latipes,latipes,latipes latipes,100.000,561.3,562.0,0.7,2,31,MAEEWGKQVFAARRHNEDTTRGSAFTYTNSNHTRDPFEGPNYHIAP...,LWS
3,AB223052.1,Mollusca,Unknown,Gastropoda,latipes,latipes,latipes latipes,100.000,561.3,561.0,0.3,2,38,MAEQWGKQVFAARRQNEDTTRGSAFTYTNSNHTRDPFEGPNYHIAP...,LWS
4,P41592.1,Chordata,Craniata,Lepidosauria,Anolis,carolinensis,Anolis carolinensis,86.179,562.9,561.0,1.9,1,12,MAGTVTEAWDVAVFAARRRNDEDDTTRDSLFTYTNSNNTRGPFEGP...,RecName: Full=Red-sensitive opsin; AltName: Fu...


In [119]:
final_err_filtered_df.shape

(50, 15)

#### <font color=#c994c7>Generate dataframe of all in-vivo values that did not match any sequences</font>

In [120]:
no_match_in_vivo_df = comp_db.drop(labels=final_err_filtered_df['comp_db_id'], inplace=False).copy()
no_match_in_vivo_df.shape

(0, 4)

In [121]:
no_match_in_vivo_df.head()

,Full_Species,Accession,Seq_Id,LambdaMax
comp_db_id,,,,


In [122]:
no_match_in_vivo_df.to_csv(f'./{query_report_dir}/no_match_in_vivo_mnm.csv', index=True)

#### <font color=#c994c7>Generate dataframe of all sequence data that did not match any in-vivo Lmax values</font>

In [123]:
no_match_seq_data_df = ncbi_filtered[~ncbi_filtered['Accession'].isin(final_err_filtered_df['Accession'])].copy()
no_match_seq_data_df.shape

(171, 13)

In [124]:
no_match_seq_data_df.head()

,level_0,index,Accession,Phylum,Subphylum,Class,Genus,Species,Full_Species,Protein,Gene_Description,Species_Synonym_Used,Prot_Len
1,1,2,P28683.1,Chordata,Craniata,Aves,Gallus,gallus,Gallus gallus,MNGTEGINFYVPMSNKTGVVRSPFEYPQYYLAEPWKYRLVCCYIFF...,RecName: Full=Green-sensitive opsin; AltName: ...,NaN,355
2,2,3,NP_990769.1,Chordata,Craniata,Aves,Gallus,gallus,Gallus gallus,MSSDDDFYLFTNGSVPGPWDGPQYHIAPPWAFYLQTAFMGIVFAVG...,violet-sensitive opsin [Gallus gallus],NaN,347
4,4,5,NP_001384426.1,Chordata,Craniata,Aves,Gallus,gallus,Gallus gallus,MNGTEGQDFYVPMSNKTGVVRSPFEYPQYYLAEPWKFSALAAYMFM...,rhodopsin [Gallus gallus],NaN,351
5,5,8,NP_990771.2,Chordata,Craniata,Aves,Gallus,gallus,Gallus gallus,MAAWEAAFAARRRHEEEDTTRDSVFTYTNSNNTRGPFEGPNYHIAP...,red-sensitive opsin [Gallus gallus],NaN,362
6,6,13,BAA00610.1,Chordata,Craniata,Aves,Gallus,gallus,Gallus gallus,MNGTEGQDFYVPMSNKTGVVRSPFEYPQYYLAEPWKFSALAAYMFM...,rhodopsin [Gallus gallus],NaN,351


In [125]:
no_match_seq_data_df.to_csv(f'./{query_report_dir}/no_match_seq_data_mnm.csv', index=True)

#### <font color=#c994c7>Compare Matches to Ground-Truth</font>

In [242]:
import pandas as pd
import numpy as np

def validate_protein_matches(df_mnm, df_sampled, blast_report_path, identity_threshold=95.0):
    """
    Validates protein matches using the BLAST report to bridge MnM Accessions 
    to Reference Seq_Ids for a given LambdaMax.
    
    Logic:
    1. Standardize LambdaMax values.
    2. Attempt to match rows where MnM['Accession'] matches Ref['Accession'] (exact).
    3. For remaining rows, use the BLAST report:
       - Take MnM['Accession'] (BLAST query_id).
       - Check if 'closest_match_id' exists in Reference for that LambdaMax.
       - Verify percent_identity >= threshold.
    """
    
    # 1. Standardize column names and types for comparison
    # Both files have 'Protein' and 'LambdaMax'
    # We ensure LambdaMax is a float to avoid "500" != "500.0" issues
    df_mnm['LambdaMax'] = pd.to_numeric(df_mnm['LambdaMax'], errors='coerce')
    df_sampled['LambdaMax'] = pd.to_numeric(df_sampled['LambdaMax'], errors='coerce')

    # 2. Identify unique combinations in the sampled (reference) file
    # We only need these two columns to check for existence
    reference_pairs = df_sampled[['Protein', 'LambdaMax']].drop_duplicates()

    reference_lookup = df_sampled[['Seq_Id', 'Accession', 'LambdaMax']].drop_duplicates()


    # 3. Perform a left merge with an indicator to find differences
    # We merge MnM (left) with the Reference Pairs (right)
    comparison = pd.merge(
        df_mnm, 
        reference_pairs, 
        on=['Protein', 'LambdaMax'], 
        how='left', 
        indicator=True
    )

    #print(comparison)
    # 4. Separate the data
    # 'both' means the combination exists in both files
    # 'left_only' means the MnM combination is unique and NOT in the sampled file
    df_matching = comparison[comparison['_merge'] == 'both'].drop(columns=['_merge'])
    df_potential_mismatches = comparison[comparison['_merge'] == 'left_only'].drop(columns=['_merge'])

    # 4. Process BLAST Report to "rescue" the mismatches
    if not df_potential_mismatches.empty:
        try:
            df_blast = pd.read_csv(blast_report_path)
            df_blast['percent_identity'] = pd.to_numeric(df_blast['percent_identity'], errors='coerce')
            
            # Filter for successful hits above threshold
            valid_hits = df_blast[
                (df_blast['status'] == 'Success') & 
                (df_blast['percent_identity'] >= identity_threshold)
            ].copy()
            
            # Prepare for lookup
            valid_hits['query_id'] = valid_hits['query_id'].astype(str)
            valid_hits['closest_match_id'] = valid_hits['closest_match_id'].astype(str)

            fuzzy_matches_mask = []
            
            for idx, row in df_potential_mismatches.iterrows():
                mnm_acc = str(row['Accession'])
                mnm_lmax = row['LambdaMax']
                
                # Find BLAST entries where MnM Accession was the query
                hits = valid_hits[valid_hits['query_id'] == mnm_acc]
                
                is_resolved = False
                if not hits.empty:
                    # Find what Seq_Ids are valid for this LambdaMax in the reference file
                    valid_seq_ids_for_lmax = reference_lookup[
                        reference_lookup['LambdaMax'] == mnm_lmax
                    ]['Seq_Id'].astype(str).unique()
                    
                    # If BLAST says this Accession matches one of those Seq_Ids, it's a match
                    if any(m_id in valid_seq_ids_for_lmax for m_id in hits['closest_match_id']):
                        is_resolved = True
                
                fuzzy_matches_mask.append(is_resolved)

            # Separate newly validated matches
            df_fuzzy_hits = df_potential_mismatches[fuzzy_matches_mask].copy()
            df_matching = pd.concat([df_matching, df_fuzzy_hits], ignore_index=True)
            df_unique_to_mnm = df_potential_mismatches[~np.array(fuzzy_matches_mask)].copy()
            
        except Exception as e:
            print(f"Error processing BLAST report: {e}")
            df_unique_to_mnm = df_potential_mismatches.copy()
    else:
        df_unique_to_mnm = df_potential_mismatches.copy()

    # 5. Final Cleanup
    # Drop helper columns
    cols_to_drop = ['_merge', 'Seq_Id']
    df_matching = df_matching.drop(columns=cols_to_drop, errors='ignore')
    df_unique_to_mnm = df_unique_to_mnm.drop(columns=cols_to_drop, errors='ignore')

    # Summary Statistics
    print(f"Total rows in MnM file: {len(df_mnm)}")
    print(f"Matching combinations: {len(df_matching)}")
    print(f"Unique MnM combinations (to be separated): {len(df_unique_to_mnm)}")
    
    return df_matching, df_unique_to_mnm

In [224]:
gt_file = './circularity_test/sampled_exclusive_wt_20260311_130205.tsv'
gt_df = pd.read_csv(gt_file, delimiter='\t')
gt_df

,Seq_Id,LambdaMax,Accession,Opsin_Family,Full_Species,Genus,Species,Phylum,Class,Protein,RefId
0,S234,358.0,U63972.1,SWS1,Rattus_norvegicus,Rattus,norvegicus,Chordata,Mammalia,MSGEXEFYLFQNISSVGPWDGPQYHIAPVWAFHLQAAFMGFVFFAG...,132.0
1,S684,507.0,OP722950.1,IV-LWS,Chrysochroa_mniszechii,Chrysochroa,mniszechii,Arthropoda,Insecta,MSALGEPNFAAWSAQRVMSGAFGGNYTVVDKVPPEMLYLVDHHWYQ...,418.0
2,S310,516.0,AF132042.1,MWS,Cavia_porcellus,Cavia,porcellus,Chordata,Mammalia,MAQRWGPHALSGVQAQDAYEDSTQASLFTYTNSNNTRGPFEGPNYH...,155.0
3,S729,532.0,OK930069.1,IV-LWS,Automeris_io,Automeris,io,Arthropoda,Insecta,MTISLDPGPGLAALQAWGGQVAAYGAANQTVVDKVPPDMLHMVDAH...,416.0
4,S436,486.0,LC260050,Rh2,Oryzias_luzonensis,Oryzias,luzonensis,Chordata,Actinopteri,MGWDGGEQNGTEGKNFYIPMSNRTGVVRSPYEYPQYYMVDPIMFKI...,367.0
5,S755,539.0,AB081277.2,LWS,Aotus_azaraiboliviensis,Aotus,azaraiboliviensis,Chordata,Lepidosauria,MAQQWSLQRLAGRHPQDNHEDSTQSSIFTYTNSNSTRGPFEGPNYH...,384.0
6,S19,498.0,AB084930.1,Rh1,Cyprichromis_leptosoma,Cyprichromis,leptosoma,Chordata,Actinopteri,MANTTGIVRSPYEYPQHYLVNPAAYAALGAYMFFLMLVGFPINFLT...,163.0
7,S23,488.0,AB185221.1,Rh1,Greenwoodochromis_bellcrossi,Greenwoodochromis,bellcrossi,Chordata,Actinopteri,MTNTTGVIRSPYEYPQHYLVSPAAYAALGAYMFFLIIVGFPINFLT...,163.0
8,S20,502.0,AB084931.1,Rh1,Dimidiochromis_compressiceps,Dimidiochromis,compressiceps,Chordata,Actinopteri,MVNTTGIVRSPYEYPQHYLVSPAAYAALGAYMFFLILVGFPINFLT...,163.0
9,S10,499.0,U57539.1,Rh1,Myripristis_violacea,Myripristis,violacea,Chordata,Actinopteri,TEGPYFYIPMSNATGIVRSPYEYPQYYLVYPAAYAVLGAYMFFLII...,160.0


In [239]:
matched_df, unique_mnm_df =validate_protein_matches(final_err_filtered_df, gt_df,"./circularity_test\mnm_on_wds-50_circ_test_2026-03-11_13-39-21\optics_on_mnm_data_2026-03-11_13-59-35/blastp_report.csv")

Total rows in MnM file: 50
Matching combinations: 47
Unique MnM combinations (to be separated): 3


In [240]:
unique_mnm_df

,Accession,Phylum,Subphylum,Class,Genus,Species,Full_Species,%Identity_Nearest_VPOD_Sequence,prediction_value,LambdaMax,abs_diff,candidate_opsin_count,comp_db_id,Protein,Gene_Description,Protein_Clean
2,AB223051.1,Mollusca,Unknown,Gastropoda,latipes,latipes,latipes latipes,100.0,561.3,562.0,0.7,2,31,MAEEWGKQVFAARRHNEDTTRGSAFTYTNSNHTRDPFEGPNYHIAP...,LWS,MAEEWGKQVFAARRHNEDTTRGSAFTYTNSNHTRDPFEGPNYHIAP...
3,AB223052.1,Mollusca,Unknown,Gastropoda,latipes,latipes,latipes latipes,100.0,561.3,561.0,0.3,2,38,MAEQWGKQVFAARRQNEDTTRGSAFTYTNSNHTRDPFEGPNYHIAP...,LWS,MAEQWGKQVFAARRQNEDTTRGSAFTYTNSNHTRDPFEGPNYHIAP...
32,BAJ60906.1,Chordata,Craniata,Actinopteri,Paracyprichromis,brieni,Paracyprichromis brieni,100.0,490.8,491.0,0.2,2,15,MNGTEGPFFYVPMANTTGIVRSPYDYPQHYLVNPAAYAALGAYMFF...,rod opsin [Paracyprichromis brieni],MNGTEGPFFYVPMANTTGIVRSPYDYPQHYLVNPAAYAALGAYMFF...


In [223]:
matched_df

,Accession,Phylum,Subphylum,Class,Genus,Species,Full_Species,%Identity_Nearest_VPOD_Sequence,prediction_value,LambdaMax,abs_diff,candidate_opsin_count,comp_db_id,Protein,Gene_Description,Protein_Clean


In [241]:

# Save the unique results to a new file
output_name = "./circularity_test/mnm_incorrect_combinations.csv"
unique_mnm_df.to_csv(output_name, index=False)
output_name = "./circularity_test/mnm_matched_combinations.csv"
matched_df.to_csv(output_name, index=False)

print(f"\nSuccess! Separated data saved to: {output_name}")


Success! Separated data saved to: ./circularity_test/mnm_matched_combinations.csv
